<a href="https://colab.research.google.com/github/kishortecarch-ai/Pokhara-seismic-ml-app/blob/main/web_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RC seismic thesis — corrected Colab launcher

Matches the **October 2026 final research report**: 144 development cases from six buildings; frozen nine-feature RF predicting **simulated MIDR (%)**, then five **provisional** threshold states. This does not retrain the model, certify structural safety, or claim calibrated fragility probabilities.

Run the cells in order. Source model and CSV must already be available in your own mounted Google Drive. **Do not upload or unzip an unrelated package.** Individual out-of-training-range warnings remain visible.

In [19]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit", "pandas", "numpy", "joblib", "scikit-learn"], check=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Dependencies installed and Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dependencies installed and Drive mounted.


In [20]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit", "pandas", "numpy", "joblib", "scikit-learn"], check=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Dependencies installed and Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dependencies installed and Drive mounted.


In [21]:
from pathlib import Path
import json, shutil, joblib
import pandas as pd

mydrive = Path("/content/drive/MyDrive")
if not mydrive.exists():
    raise FileNotFoundError("Google Drive not mounted at /content/drive/MyDrive")
# Search both MyDrive/Pokhara_RC_Seismic_ML and MyDrive/Thesis/Pokhara_RC_Seismic_ML.
# Search is read-only; do not overwrite archived research files.
model_candidates = sorted(mydrive.rglob("06_selected_exploratory_model.joblib"))
data_candidates = sorted(mydrive.rglob("01_original_144_labels_geometry.csv"))
if not model_candidates or not data_candidates:
    raise FileNotFoundError(f"Missing frozen model or 144-case CSV. Models={model_candidates}, data={data_candidates}")

def pick(paths):
    filtered = [p for p in paths if "Pokhara_RC_Seismic_ML" in p.parts]
    return (filtered or paths)[0]
model_path, data_path = pick(model_candidates), pick(data_candidates)
print("Frozen model:", model_path)
print("144-case training data:", data_path)

pkg = joblib.load(model_path)  # Open only your own trusted thesis model.
expected = ["target_PGA_g", "direction", "column_width_mm", "column_depth_mm",
            "beam_width_mm", "beam_depth_mm", "storeys_from_frame",
            "height_from_frame_m", "plan_bbox_area_m2"]
if not isinstance(pkg, dict) or not all(k in pkg for k in ["features", "pipeline", "direction_encoding"]):
    raise ValueError("This is not the archived frozen thesis model package")
if list(pkg["features"]) != expected:
    raise ValueError(f"Unexpected feature contract: {pkg['features']!r}")
if pkg["direction_encoding"] != {"X": 0, "Y": 1}:
    raise ValueError(f"Direction coding mismatch: {pkg['direction_encoding']!r}")
train = pd.read_csv(data_path)
if len(train) != 144:
    raise ValueError(f"Expected 144 original cases, got {len(train)}")
train["direction"] = train["direction"].replace({"X": 0, "Y": 1})
for col in expected:
    if col not in train.columns: raise KeyError(f"Missing training feature: {col}")
    train[col] = pd.to_numeric(train[col], errors="coerce")
    if not train[col].notna().any(): raise ValueError(f"All values missing: {col}")

app_dir = Path("/content/rc_seismic_thesis_app")
app_dir.mkdir(parents=True, exist_ok=True)
# This companion app code can be uploaded to Colab using the Files sidebar.
# In the distributed notebook, the app source is embedded in a later cell.
meta = {"frozen_model_path": str(model_path), "feature_order": expected,
        "training_feature_ranges": {c: {"min": float(train[c].min()), "max": float(train[c].max())} for c in expected},
        "development_cases": 144, "development_buildings": 6}
(app_dir / "model_input_bounds.json").write_text(json.dumps(meta, indent=2))
print("Verified contract; calculated real training ranges. App directory:", app_dir)

Frozen model: /content/drive/MyDrive/Thesis/Pokhara_RC_Seismic_ML/results/phase75_nested_LOBO_model_selection/06_selected_exploratory_model.joblib
144-case training data: /content/drive/MyDrive/Thesis/Pokhara_RC_Seismic_ML/results/phase69_master_dataset_integration/01_original_144_labels_geometry.csv
Verified contract; calculated real training ranges. App directory: /content/rc_seismic_thesis_app


/tmp/ipykernel_1549/2087503445.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train["direction"] = train["direction"].replace({"X": 0, "Y": 1})


In [24]:
from pathlib import Path
app_source = '"""Research-only Streamlit app for the frozen nine-feature thesis model.\nOutputs are simulated MIDR, not verified damage or structural safety.\n"""\nimport json\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\nHERE = Path(__file__).resolve().parent\nmeta = json.loads((HERE / "model_input_bounds.json").read_text())\nfeature_order = meta["feature_order"]\n\n@st.cache_resource(show_spinner="Loading frozen thesis model...")\ndef get_package():\n    pkg = joblib.load(meta["frozen_model_path"])\n    if not isinstance(pkg, dict) or "pipeline" not in pkg or "features" not in pkg:\n        raise ValueError("Archived model is not the expected thesis package.")\n    if list(pkg["features"]) != feature_order:\n        raise ValueError("Archived model feature order does not match the saved contract.")\n    encoding = pkg.get("direction_encoding")\n    if encoding != {"X": 0, "Y": 1}:\n        raise ValueError(f"Unexpected direction encoding: {encoding!r}; expected X=0, Y=1")\n    return pkg\n\nst.set_page_config(page_title="RC seismic simulated drift | MIT-AI", layout="centered")\nst.title("Rapid seismic response — RC research prototype")\nst.warning("RESEARCH ONLY: simulated MIDR is not observed damage, collapse probability, or an engineering safety approval.")\nst.caption("Based on six development buildings / 144 nonlinear simulation cases; one scaled El Centro record. Nine-feature frozen Random Forest; no new training.")\n\npkg = get_package()\nbounds = meta["training_feature_ranges"]\n\ndef numeric(k, label, step=1.0, integer=False):\n    b = bounds[k]\n    mid = (b["min"] + b["max"]) / 2\n    if integer:\n        return st.number_input(label, min_value=1, value=max(1, int(round(mid))), step=1)\n    return st.number_input(label, min_value=0.0, value=float(mid), step=float(step), format="%.4f")\n\nwith st.form("single_case"):\n    pga = numeric("target_PGA_g", "Target PGA (g)", 0.05)\n    direction = st.selectbox("Loading direction", ["X", "Y"])\n    cw = numeric("column_width_mm", "Representative column width (mm)", 10)\n    cd = numeric("column_depth_mm", "Representative column depth (mm)", 10)\n    bw = numeric("beam_width_mm", "Representative beam width (mm)", 10)\n    bd = numeric("beam_depth_mm", "Representative beam depth (mm)", 10)\n    storeys = numeric("storeys_from_frame", "Storeys", integer=True)\n    height = numeric("height_from_frame_m", "Frame height (m)", 0.5)\n    area = numeric("plan_bbox_area_m2", "Plan bounding-box area (m², not net floor area)", 5)\n    submit = st.form_submit_button("Predict simulated MIDR", type="primary")\n\nif submit:\n    values = {\n        "target_PGA_g": pga, "direction": pkg["direction_encoding"][direction],\n        "column_width_mm": cw, "column_depth_mm": cd,\n        "beam_width_mm": bw, "beam_depth_mm": bd,\n        "storeys_from_frame": storeys, "height_from_frame_m": height,\n        "plan_bbox_area_m2": area,\n    }\n    if any(not np.isfinite(v) for v in values.values()) or any(values[k] <= 0 for k in values if k != "direction"):\n        st.error("All physical quantities, including PGA, must be finite and positive.")\n        st.stop()\n    outside = [(k, v, bounds[k]) for k, v in values.items()\n               if v < bounds[k]["min"] - 1e-9 or v > bounds[k]["max"] + 1e-9]\n    if outside:\n        st.error("OUTSIDE TRAINING RANGE — the following inputs differ from the values observed in the 144 development cases:")\n        for k, v, b in outside:\n            st.write(f"- {k}: entered {v:g}; training range {b[\'min\']:g} to {b[\'max\']:g}")\n        st.caption("Prediction is shown as a research demonstration only, with elevated extrapolation risk.")\n    else:\n        st.info("All individual inputs fall inside observed ranges; this does NOT guarantee that their combination or building geometry is represented in training.")\n\n    sample = pd.DataFrame([[values[k] for k in feature_order]], columns=feature_order)\n    y = float(pkg["pipeline"].predict(sample)[0])\n    # Thresholds in MIDR %, matching the final thesis (not calibrated fragility probabilities).\n    thresholds = [0.5, 1.0, 3.0, 8.0]\n    labels = ["None", "Slight", "Moderate", "Extensive", "Complete"]\n    cls = labels[int(np.searchsorted(thresholds, y, side="right"))]\n    st.metric("Predicted simulated MIDR", f"{y:.3f}%")\n    st.write(f"**Provisional ordinal drift state:** {cls}")\n    st.caption("The training classification set had NO Complete examples; a regression-derived state is not a validated damage classification.")\n    st.caption("Final thesis: nested LOBO R²=0.8671 on six buildings; external R²=-0.0083 on five unseen buildings. Treat unfamiliar geometry, especially irregular plans, with caution.")\n'
Path('/content/rc_seismic_thesis_app/app.py').write_text(app_source, encoding='utf-8')
print('App created from the report-aligned frozen model contract')

App created from the report-aligned frozen model contract


In [23]:
# Optional archived reproducibility test, if available in Drive.
from pathlib import Path
import subprocess, sys
hits = sorted(Path("/content/drive/MyDrive").rglob("test_frozen_model.py"))
if hits:
    chosen = next((p for p in hits if "Pokhara_RC_Seismic_ML" in p.parts), hits[0])
    print("Running archived test:", chosen)
    res = subprocess.run([sys.executable, str(chosen)], cwd=str(chosen.parent), capture_output=True, text=True)
    print((res.stdout + "\n" + res.stderr)[-5000:])
    if res.returncode: print("Archived verification did not pass in this runtime; check package and saved inputs.")
else:
    print("Optional archived test_frozen_model.py not found. No 112/112 verification claim is made in this runtime.")

Optional archived test_frozen_model.py not found. No 112/112 verification claim is made in this runtime.


In [22]:
import subprocess, sys, time, urllib.request
from pathlib import Path
from google.colab import output
app_file = Path("/content/rc_seismic_thesis_app/app.py")
if not app_file.is_file(): raise FileNotFoundError(app_file)
log_path = Path("/content/rc_seismic_streamlit.log")
log_file = log_path.open("w")
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(app_file),
                         "--server.port=8501", "--server.address=127.0.0.1",
                         "--server.headless=true"], stdout=log_file, stderr=subprocess.STDOUT)
for _ in range(25):
    if proc.poll() is not None: break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=1) as resp:
            if resp.status == 200: break
    except Exception: time.sleep(1)
else:
    raise RuntimeError("App did not become healthy. Logs:\n" + log_path.read_text()[-3000:])
if proc.poll() is not None:
    raise RuntimeError("Streamlit exited. Logs:\n" + log_path.read_text()[-3000:])
print("Streamlit is running. Use the embedded Colab preview below. Keep runtime active.")
output.serve_kernel_port_as_iframe(8501, height=800)

Streamlit is running. Use the embedded Colab preview below. Keep runtime active.


<IPython.core.display.Javascript object>